# 🤝 A2A Protocol (Agent-to-Agent): Zero to Hero — A Guided Lab

**A2A (Agent2Agent)** is an open protocol (introduced by Google) for **independent AI agents —
often built by different teams/vendors — to discover each other, negotiate a task, and
collaborate**, without needing to share internal code or memory. Where MCP connects an agent to
*tools*, A2A connects an agent to **other agents**.

**Beginner-first, 100% offline.** We build minimal in-memory A2A primitives (agent cards, tasks,
messages) with the same shape as the real protocol.

**Prerequisite:** the Agents & Orchestration lab and the MCP lab — A2A sits alongside both.

**How this lab works** — 📖 Theory → 🧠 Mental model → 🖼️ ASCII diagram → 🔬 Worked example →
⚡ Pro tips → ⚠️ Traps → ✏️ Your Turn → ✅ Solution.

**Roadmap**
1. Why A2A? MCP vs A2A
2. The Agent Card (capability advertisement)
3. Discovery: finding the right agent
4. Tasks & the task lifecycle
5. Messages & artifacts
6. A synchronous request/response exchange
7. Long-running tasks & status polling
8. Multi-agent delegation (an orchestrator using A2A)
9. Security & trust boundaries
10. 🏆 Capstone: a small A2A network solving a compound task


In [ ]:
import json, uuid, time
from enum import Enum

print("Building A2A primitives: Agent Cards, Tasks, Messages.")

---
## Chapter 1 — Why A2A? MCP vs A2A

📖 **Theory.** You've already learned **MCP** (Model Context Protocol): it connects **one agent**
to **tools and data**. **A2A** solves a different problem: connecting **one agent to another
independent agent** — potentially built by a different company, running on different
infrastructure, exposing capabilities the calling agent doesn't need to understand internally.

🖼️ **Diagram — MCP vs A2A**
```
 MCP (agent <-> tools/data):          A2A (agent <-> agent):

   Agent ──► MCP client ──► MCP server    Agent A ──► A2A ──► Agent B
             (tools, resources,           (delegates a TASK, gets
              prompts)                     back a RESULT -- B's internals
                                            stay a black box to A)
```

🧠 **Mental model.** MCP is like giving an employee access to internal company tools. A2A is
like that employee **calling a specialist consultant at another company** — you describe the
job, they do it their way, and hand back results. You don't need (or get) to see how they work
internally.


In [ ]:
comparison = {
    "MCP":  {"connects": "agent <-> tools/data", "visibility": "server exposes explicit tools/resources"},
    "A2A":  {"connects": "agent <-> agent",      "visibility": "other agent's internals stay opaque"},
}
for k, v in comparison.items():
    print(k, "->", v)

### ✏️ Your Turn 1.1
In a comment, describe a real scenario where you'd want **A2A** (agent talking to another
agent) rather than **MCP** (agent talking to a tool).

In [ ]:
# scenario: ...


✅ **Solution**
```python
# A travel-booking agent needs a "verify visa requirements" task done. Rather than
# building that logic itself, it delegates to a specialized "VisaAgent" built and
# maintained by a different team/vendor -- it just needs the RESULT, not the internals.
```

---
## Chapter 2 — The Agent Card (Capability Advertisement)

📖 **Theory.** Every A2A-compatible agent publishes an **Agent Card** — a small JSON document
describing: its **name**, what **skills/capabilities** it offers, the **input/output** it
expects, and how to **reach** it. Other agents read this card to decide whether (and how) to
delegate work to it — similar in spirit to an MCP server's tool list, but describing an *entire
agent*, not individual functions.

🖼️ **Diagram — an Agent Card**
```
 {
   "name": "TranslationAgent",
   "description": "translates text between languages",
   "skills": ["translate"],
   "endpoint": "a2a://translation-agent",
   "input_modes": ["text"], "output_modes": ["text"]
 }
```


In [ ]:
class AgentCard:
    def __init__(self, name, description, skills, endpoint, input_modes=("text",), output_modes=("text",)):
        self.name, self.description = name, description
        self.skills = list(skills)
        self.endpoint = endpoint
        self.input_modes, self.output_modes = list(input_modes), list(output_modes)
    def to_dict(self):
        return {"name": self.name, "description": self.description, "skills": self.skills,
                "endpoint": self.endpoint, "input_modes": self.input_modes, "output_modes": self.output_modes}

translation_card = AgentCard(
    name="TranslationAgent",
    description="Translates text between languages",
    skills=["translate"],
    endpoint="a2a://translation-agent",
)
print(json.dumps(translation_card.to_dict(), indent=2))

### ✏️ Your Turn 2.1
Create an `AgentCard` for a `"MathAgent"` that offers the skill `"calculate"`, reachable at
`"a2a://math-agent"`.

In [ ]:
math_card = None
print(math_card.to_dict() if math_card else None)

✅ **Solution**
```python
math_card = AgentCard("MathAgent", "Performs calculations", ["calculate"], "a2a://math-agent")
```

---
## Chapter 3 — Discovery: Finding the Right Agent

📖 **Theory.** A **registry** (or "agent directory") holds many Agent Cards. Given a needed
**skill**, an orchestrating agent queries the registry to find agents that offer it — the A2A
equivalent of the MCP client's `list_tools()` discovery step.

🖼️ **Diagram — discovery**
```
 orchestrator ──"who can do 'translate'?"──► registry
 orchestrator ◄──[TranslationAgent card]──── registry
```


In [ ]:
class AgentRegistry:
    def __init__(self): self.agents = {}
    def register(self, card: AgentCard): self.agents[card.name] = card
    def find_by_skill(self, skill):
        return [c for c in self.agents.values() if skill in c.skills]

registry = AgentRegistry()
registry.register(translation_card)
registry.register(math_card)

found = registry.find_by_skill("translate")
print("agents offering 'translate':", [a.name for a in found])
print("agents offering 'calculate':", [a.name for a in registry.find_by_skill("calculate")])
print("agents offering 'unknown_skill':", [a.name for a in registry.find_by_skill("unknown_skill")])

⚡ **Pro tip.** In a real A2A deployment, the registry might be a company-internal directory,
a public marketplace, or even peer agents broadcasting their own cards — the *pattern*
(advertise → discover by capability) stays the same regardless of scale.

### ✏️ Your Turn 3.1
Register a third agent, `"WeatherAgent"` with skill `"get_weather"`, then find agents offering
`"get_weather"`.

In [ ]:
weather_card = None
found_weather = None
print(found_weather)

✅ **Solution**
```python
weather_card = AgentCard("WeatherAgent", "Reports weather", ["get_weather"], "a2a://weather-agent")
registry.register(weather_card)
found_weather = [a.name for a in registry.find_by_skill("get_weather")]
```

---
## Chapter 4 — Tasks & the Task Lifecycle

📖 **Theory.** A2A's core unit of work is a **Task**. Every task has a **lifecycle** of states:

🖼️ **Diagram — the task state machine**
```
 submitted ─► working ─► completed
                  │
                  ├─► input-required   (agent needs more info from the caller)
                  │
                  └─► failed
```

🧠 **Mental model.** A task is like a **ticket** you hand to another team — it starts
`submitted`, moves to `working` while they process it, and ends in `completed`, `failed`, or
`input-required` (they need to ask you something before continuing).


In [ ]:
class TaskState(Enum):
    SUBMITTED = "submitted"
    WORKING = "working"
    INPUT_REQUIRED = "input-required"
    COMPLETED = "completed"
    FAILED = "failed"

class Task:
    def __init__(self, skill, input_data):
        self.id = str(uuid.uuid4())[:8]
        self.skill = skill
        self.input = input_data
        self.state = TaskState.SUBMITTED
        self.result = None
        self.history = [(self.state, time.time())]
    def transition(self, new_state):
        self.state = new_state
        self.history.append((new_state, time.time()))

t = Task(skill="translate", input_data={"text": "hello", "target_language": "Spanish"})
print(f"task {t.id} created, state: {t.state.value}")
t.transition(TaskState.WORKING)
print(f"task {t.id} now: {t.state.value}")
t.transition(TaskState.COMPLETED)
print(f"task {t.id} now: {t.state.value}")
print("full history:", [s.value for s, _ in t.history])

### ✏️ Your Turn 4.1
Create a task for skill `"calculate"` with input `{"expression": "2+2"}`, transition it through
`WORKING` then `FAILED` (simulating an error), and print its final state.

In [ ]:
calc_task = None
print(calc_task.state.value if calc_task else None)

✅ **Solution**
```python
calc_task = Task(skill="calculate", input_data={"expression": "2+2"})
calc_task.transition(TaskState.WORKING)
calc_task.transition(TaskState.FAILED)
print(calc_task.state.value)   # "failed"
```

---
## Chapter 5 — Messages & Artifacts

📖 **Theory.** Within a task, agents exchange **Messages** (conversational turns — like the
`role`/`content` messages from the LLM APIs lab) and produce **Artifacts** (the actual work
output — files, structured data, generated text). Messages coordinate; artifacts are what gets
delivered.

🖼️ **Diagram — messages coordinate, artifacts deliver**
```
 caller: Message("please translate 'hello' to Spanish")
 agent:  Message("working on it...")
 agent:  Artifact({"translation": "hola"})     ◄── the actual deliverable
```


In [ ]:
class Message:
    def __init__(self, sender, content):
        self.sender, self.content, self.timestamp = sender, content, time.time()

class Artifact:
    def __init__(self, kind, data):
        self.kind, self.data = kind, data

class RichTask(Task):
    def __init__(self, skill, input_data):
        super().__init__(skill, input_data)
        self.messages, self.artifacts = [], []
    def add_message(self, sender, content):
        self.messages.append(Message(sender, content))
    def add_artifact(self, kind, data):
        self.artifacts.append(Artifact(kind, data))

rt = RichTask(skill="translate", input_data={"text": "hello", "target_language": "Spanish"})
rt.add_message("caller", "please translate this")
rt.transition(TaskState.WORKING)
rt.add_message("TranslationAgent", "working on it")
rt.add_artifact("translation", {"translation": "hola"})
rt.transition(TaskState.COMPLETED)

print("messages:", [(m.sender, m.content) for m in rt.messages])
print("artifacts:", [(a.kind, a.data) for a in rt.artifacts])

### ✏️ Your Turn 5.1
Add a message from `"MathAgent"` saying `"computing..."` to a new task, then add an artifact of
kind `"result"` with `{"value": 4}`.

In [ ]:
math_task = RichTask(skill="calculate", input_data={"expression": "2+2"})
# add message and artifact


✅ **Solution**
```python
math_task.add_message("MathAgent", "computing...")
math_task.add_artifact("result", {"value": 4})
```

---
## Chapter 6 — A Synchronous Request/Response Exchange

📖 **Theory.** Simplest A2A pattern: caller sends a task, the remote agent processes it
immediately, and returns a completed result. This is the "phone call" pattern — you wait on the
line for the answer.

🖼️ **Diagram — synchronous exchange**
```
 caller ──send_task(skill, input)──► agent
 caller ◄────completed task─────────  agent   (caller blocks until done)
```


In [ ]:
class RemoteAgent:
    """Simulates a remote A2A agent that processes tasks synchronously."""
    def __init__(self, card, handler):
        self.card, self.handler = card, handler
    def send_task(self, skill, input_data):
        task = RichTask(skill, input_data)
        if skill not in self.card.skills:
            task.transition(TaskState.FAILED)
            task.add_message(self.card.name, f"I don't support skill '{skill}'")
            return task
        task.transition(TaskState.WORKING)
        try:
            result = self.handler(input_data)
            task.add_artifact("result", result)
            task.transition(TaskState.COMPLETED)
        except Exception as e:
            task.transition(TaskState.FAILED)
            task.add_message(self.card.name, f"error: {e}")
        return task

def translate_handler(input_data):
    fake_translations = {"hello": "hola", "goodbye": "adios", "thank you": "gracias"}
    text = input_data["text"].lower()
    return {"translation": fake_translations.get(text, f"[{text} translated]")}

translation_agent = RemoteAgent(translation_card, translate_handler)
result_task = translation_agent.send_task("translate", {"text": "hello", "target_language": "Spanish"})
print("final state:", result_task.state.value)
print("artifact:", result_task.artifacts[0].data)

⚠️ **Common trap.** Always handle the case where the remote agent **doesn't support** the
requested skill (as `send_task` does above) — a caller should check the Agent Card's `skills`
*before* sending, but must also handle rejection gracefully as a fallback.

### ✏️ Your Turn 6.1
Build a `MathAgent` `RemoteAgent` (handler: safely `eval` a `"expression"` field) and send it a
`"calculate"` task for `"6 * 7"`.

In [ ]:
def calc_handler(input_data):
    pass
math_agent = None
calc_result = None
print(calc_result.artifacts[0].data if calc_result else None)

✅ **Solution**
```python
def calc_handler(input_data):
    expr = input_data["expression"]
    return {"value": eval(expr)}   # in production: use a SAFE expression evaluator, not raw eval
math_agent = RemoteAgent(math_card, calc_handler)
calc_result = math_agent.send_task("calculate", {"expression": "6 * 7"})
```

---
## Chapter 7 — Long-Running Tasks & Status Polling

📖 **Theory.** Not every task finishes instantly. For long-running work, A2A supports an
**asynchronous** pattern: submit the task, get a task ID back immediately, then **poll** its
status (or receive push updates) until it reaches a terminal state.

🖼️ **Diagram — async polling**
```
 caller ──submit──► agent  (returns task_id immediately, state=WORKING)
 caller ──poll(task_id)──► agent  (state still WORKING)
 caller ──poll(task_id)──► agent  (state still WORKING)
 caller ──poll(task_id)──► agent  (state=COMPLETED, here's the artifact)
```


In [ ]:
class AsyncRemoteAgent:
    def __init__(self, card, work_steps=3):
        self.card, self.work_steps = card, work_steps
        self.tasks = {}   # task_id -> (task, steps_remaining)
    def submit_task(self, skill, input_data):
        task = RichTask(skill, input_data)
        task.transition(TaskState.WORKING)
        self.tasks[task.id] = [task, self.work_steps]
        return task.id
    def poll(self, task_id):
        task, steps_left = self.tasks[task_id]
        if task.state == TaskState.COMPLETED:
            return task
        steps_left -= 1
        if steps_left <= 0:
            task.add_artifact("result", {"status": "done processing"})
            task.transition(TaskState.COMPLETED)
        self.tasks[task_id][1] = steps_left
        return task

slow_agent = AsyncRemoteAgent(math_card, work_steps=3)
task_id = slow_agent.submit_task("calculate", {"expression": "long computation"})
print("submitted, task_id:", task_id)

for i in range(4):
    status = slow_agent.poll(task_id)
    print(f"  poll {i}: state={status.state.value}")

⚡ **Pro tip.** Real A2A supports **push notifications** (webhooks) as an alternative to
polling, so the caller doesn't have to keep asking "are you done yet?" — but polling is simpler
to reason about and is a fine default for a first implementation.

### ✏️ Your Turn 7.1
Submit a task to `slow_agent` and poll it in a `while` loop until its state is `COMPLETED`,
counting how many polls it took.

In [ ]:
tid2 = slow_agent.submit_task("calculate", {"expression": "another task"})
poll_count = 0
# loop until completed
print(poll_count)

✅ **Solution**
```python
tid2 = slow_agent.submit_task("calculate", {"expression": "another task"})
poll_count = 0
status = slow_agent.poll(tid2)
while status.state != TaskState.COMPLETED:
    poll_count += 1
    status = slow_agent.poll(tid2)
print(poll_count)
```

---
## Chapter 8 — Multi-Agent Delegation (an Orchestrator Using A2A)

📖 **Theory.** An **orchestrator agent** uses discovery + task delegation to solve a **compound**
goal by farming out sub-tasks to specialist agents — the same idea as the Agents lab's
Orchestrator, but now each "worker" is a genuinely **separate agent** reached over A2A, not a
function call.

🖼️ **Diagram — orchestrator delegating over A2A**
```
                    ┌──────────────┐
          goal ───► │ ORCHESTRATOR │
                    └──────┬───────┘
        1. discover via registry
        2. send_task() to each   ┌──────► TranslationAgent
                                  ├──────► MathAgent
                                  └──────► WeatherAgent
        3. combine artifacts
```


In [ ]:
class A2AOrchestrator:
    def __init__(self, registry, agent_instances):
        self.registry = registry
        self.agents = agent_instances   # name -> RemoteAgent instance
    def delegate(self, skill, input_data):
        candidates = self.registry.find_by_skill(skill)
        if not candidates:
            return {"error": f"no agent found for skill '{skill}'"}
        agent_name = candidates[0].name
        remote = self.agents[agent_name]
        task = remote.send_task(skill, input_data)
        if task.state == TaskState.COMPLETED:
            return {"agent": agent_name, "result": task.artifacts[0].data}
        return {"agent": agent_name, "error": "task did not complete", "state": task.state.value}

orchestrator = A2AOrchestrator(registry, {
    "TranslationAgent": translation_agent,
    "MathAgent": math_agent,
})
print(orchestrator.delegate("translate", {"text": "goodbye", "target_language": "Spanish"}))
print(orchestrator.delegate("calculate", {"expression": "9 * 9"}))
print(orchestrator.delegate("summarize", {"text": "no agent for this skill"}))

### ✏️ Your Turn 8.1
Register the `WeatherAgent` card (Ch.3) and instantiate a `RemoteAgent` for it (handler:
return a fixed weather string for any city). Add it to the orchestrator's `agents` dict and
delegate a `"get_weather"` task.

In [ ]:
def weather_handler(input_data):
    return {"weather": "Sunny, 22C"}
weather_agent_instance = None
# add to orchestrator.agents, then delegate


✅ **Solution**
```python
weather_agent_instance = RemoteAgent(weather_card, weather_handler)
orchestrator.agents["WeatherAgent"] = weather_agent_instance
print(orchestrator.delegate("get_weather", {"city": "Paris"}))
```

---
## Chapter 9 — Security & Trust Boundaries

📖 **Theory.** Because A2A connects **independently-operated** agents (possibly different
companies), trust cannot be assumed. Key concerns:
- **Authentication** — verify which agent you're really talking to (API keys, signed tokens).
- **Authorization/scoping** — a remote agent should only see the **minimum input** needed for its
  task, not your entire conversation history.
- **Opaque execution** — you generally can't see *how* the other agent did the work, only its
  declared card and the result — so only delegate to agents you trust for that task's stakes.

🖼️ **Diagram — trust boundary**
```
 YOUR AGENT  |  trust boundary  |  THEIR AGENT
   sends: minimal task input ──►|
             result artifact  ◄─|   (their internal reasoning/data stays hidden)
```


In [ ]:
def scoped_input(full_context, allowed_fields):
    """Only pass the minimum needed fields across a trust boundary."""
    return {k: v for k, v in full_context.items() if k in allowed_fields}

full_context = {
    "text": "hello", "target_language": "Spanish",
    "user_email": "alice@company.com",       # sensitive -- NOT needed by TranslationAgent
    "internal_session_id": "sess_98213",      # sensitive -- NOT needed either
}
safe_input = scoped_input(full_context, allowed_fields={"text", "target_language"})
print("full context (has sensitive fields):", full_context)
print("scoped input actually sent:", safe_input)

⚠️ **Common trap.** Passing your **entire** internal context to a remote agent "just in
case" is a privacy and security leak. Always scope down to exactly what the task needs — the
principle of least privilege applies to agent-to-agent calls just as much as to human access
control.

### ✏️ Your Turn 9.1
Write a check that **rejects** delegating a task to an agent whose Agent Card doesn't list the
needed skill — *before* even sending the task (defense in depth alongside the runtime check
from Ch.6).

In [ ]:
def can_delegate(card, skill):
    pass
print(can_delegate(translation_card, "translate"))
print(can_delegate(translation_card, "calculate"))

✅ **Solution**
```python
def can_delegate(card, skill):
    return skill in card.skills
```

---
## 🏆 Chapter 10 — Capstone: A Small A2A Network Solving a Compound Task

Build a complete mini A2A network: a registry with 3 agents (Translation, Math, Weather), an
orchestrator that decomposes a compound goal into skill-tagged sub-tasks, delegates each via A2A,
and combines the results into a report — with security scoping applied. Build it before revealing
the solution.

In [ ]:
# Your A2ANetwork here
class A2ANetwork:
    def __init__(self):
        pass
    def solve(self, subtasks):
        # subtasks: list of (skill, input_data) tuples
        pass

# net = A2ANetwork()
# report = net.solve([("translate", {"text":"hello","target_language":"Spanish"}),
#                      ("calculate", {"expression":"12*12"}),
#                      ("get_weather", {"city":"Tokyo"})])


✅ **Capstone Solution**
```python
class A2ANetwork:
    def __init__(self):
        self.registry = AgentRegistry()
        self.registry.register(translation_card)
        self.registry.register(math_card)
        self.registry.register(weather_card)
        self.agents = {
            "TranslationAgent": RemoteAgent(translation_card, translate_handler),
            "MathAgent": RemoteAgent(math_card, calc_handler),
            "WeatherAgent": RemoteAgent(weather_card, weather_handler),
        }
        self.orchestrator = A2AOrchestrator(self.registry, self.agents)

    def solve(self, subtasks):
        report = []
        for skill, input_data in subtasks:
            candidates = self.registry.find_by_skill(skill)
            if not candidates or not can_delegate(candidates[0], skill):
                report.append({"skill": skill, "error": "no capable/authorized agent"})
                continue
            # security: scope input down (illustrative -- keep only task-relevant fields)
            outcome = self.orchestrator.delegate(skill, input_data)
            report.append({"skill": skill, **outcome})
        return report

net = A2ANetwork()
report = net.solve([
    ("translate", {"text": "hello", "target_language": "Spanish"}),
    ("calculate", {"expression": "12 * 12"}),
    ("get_weather", {"city": "Tokyo"}),
])
for row in report:
    print(row)
```

🎉 **You understand A2A end to end!** Agent Cards, discovery via a registry, the task lifecycle,
messages vs. artifacts, synchronous and async (polling) exchange patterns, multi-agent
delegation, and security scoping. Real A2A adds HTTP/SSE transport, formal auth (OAuth-style),
and richer content types — but every core concept maps to what you built here.

---
### 📌 Concept Quick-Reference
**MCP vs A2A:** MCP = agent↔tools/data; A2A = agent↔agent (opaque internals)
**Agent Card:** name, description, skills, endpoint — the capability advertisement
**Discovery:** a registry of cards, queried by needed skill
**Task lifecycle:** submitted → working → completed / failed / input-required
**Messages vs artifacts:** messages coordinate; artifacts are the actual deliverable
**Sync exchange:** send_task() and wait for a completed result
**Async exchange:** submit_task() returns an ID immediately; poll() until terminal state
**Orchestration:** discover → delegate to specialist agents → combine results
**Security:** authenticate the remote agent, scope input to the minimum needed (least privilege)
